# 面试题：线上 A/B 实验如何处理随机化、CUPED、多指标和提前偷看？

本 Notebook 手写稳定用户分桶、SRM、difference-in-means/Welch SE、置信区间、CUPED、ratio delta method、多重检验、guardrail 非劣和 sequential alpha spending。只用 NumPy/基础统计公式，不调用实验平台或 statsmodels。

合成实验知道真实 treatment effect，用于验证估计与审计；真实上线仍需伦理、隐私、触发逻辑、日志完整性和业务决策流程。

In [ ]:
import copy,hashlib,json,math,warnings  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from statistics import NormalDist  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import numpy as np  # 导入本单元所需的依赖。
RNG78=np.random.default_rng(7801); NORMAL78=NormalDist()  # 计算并保存当前步骤的中间状态。
def canonical78(x): return json.dumps(x,sort_keys=True,separators=(",",":"))  # 定义本节可复用的核心函数。
def sha78(x): return hashlib.sha256(x).hexdigest()  # 定义本节可复用的核心函数。
assert math.isclose(NORMAL78.cdf(0),.5)  # 用受控断言验证关键不变量。

## 1. 随机化单位与稳定分桶

以 user 为随机化单位，`hash(experiment_id:salt:user_id)` 映射到 `[0,1)`；同一用户跨设备事件保持同组。实验 ID/salt 不同会得到不同分配。不能用 Python hash 或每次请求随机，否则串组。

先定义 eligible/trigger，再分桶；分析单位与随机化单位不一致时标准误要 cluster 化。

In [ ]:
def bucket78(user_id,experiment_id="search-ui-v1",salt="s1"):  # 定义本节可复用的核心函数。
    if not user_id or not experiment_id: raise ValueError("assignment_contract")  # 按当前条件选择后续控制路径。
    raw=hashlib.sha256(f"{experiment_id}:{salt}:{user_id}".encode()).digest(); return int.from_bytes(raw[:8],"big")/2**64  # 计算并保存当前步骤的中间状态。
def assign78(user_id,ratio=.5):  # 定义本节可复用的核心函数。
    if not 0<ratio<1: raise ValueError("ratio_contract")  # 按当前条件选择后续控制路径。
    return int(bucket78(user_id)<ratio)  # 返回当前分支计算出的结果。
users78=[f"user-{i:05d}" for i in range(12000)]; treatment78=np.array([assign78(u) for u in users78])  # 计算并保存当前步骤的中间状态。
assert np.array_equal(treatment78,np.array([assign78(u) for u in users78]))  # 用受控断言验证关键不变量。
assert .48<treatment78.mean()<.52 and set(treatment78)=={0,1}  # 用受控断言验证关键不变量。
assert bucket78("user-1")!=bucket78("user-1","other")  # 用受控断言验证关键不变量。
try: assign78("u",1.); raise AssertionError("invalid ratio accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="ratio_contract"  # 捕获预期异常并验证失败分支。

## 2. Sample Ratio Mismatch 是第一道门

在看效果前检查实际组人数是否符合预期。二组 50/50 可用 Pearson chi-square，再用一自由度 survival function；本例用 `erfc(sqrt(chi2/2))`。SRM 往往表示分桶、触发、日志或过滤 bug，不能继续解释 treatment effect。

大样本下极小偏差也显著，因此还要报告实际比例和诊断切片。

In [ ]:
def srm78(assignments,expected_ratio=.5):  # 定义本节可复用的核心函数。
    x=np.asarray(assignments); counts=np.array([(x==0).sum(),(x==1).sum()],float); expected=np.array([1-expected_ratio,expected_ratio])*len(x); chi=float(((counts-expected)**2/expected).sum()); p=math.erfc(math.sqrt(chi/2)); return {"counts":counts.astype(int),"chi2":chi,"p":p}  # 计算并保存当前步骤的中间状态。
srm_ok78=srm78(treatment78)  # 计算并保存当前步骤的中间状态。
assert srm_ok78["p"]>.01 and srm_ok78["counts"].sum()==len(users78)  # 用受控断言验证关键不变量。
broken_assignment78=treatment78.copy(); broken_assignment78[:800]=1; srm_bad78=srm78(broken_assignment78)  # 计算并保存当前步骤的中间状态。
assert srm_bad78["p"]<1e-6 and srm_bad78["chi2"]>srm_ok78["chi2"]  # 用受控断言验证关键不变量。
assert all(srm_ok78["counts"]>0)  # 用受控断言验证关键不变量。

## 3. 生成 pre-period、主指标和 guardrail

用户潜在活跃度同时影响 pre-period 与实验期收入，真实 treatment 对收入加 0.18；延迟 guardrail 增加 1.2ms。每用户还生成 sessions/orders，后续演示 ratio metric。

随机噪声在分配后采样，但潜在值与 treatment 独立。所有指标先聚合到 user，避免高活跃用户事件行重复加权。

In [ ]:
n78=len(users78); activity78=RNG78.normal(size=n78); pre78=3+1.7*activity78+RNG78.normal(scale=1.1,size=n78); revenue78=5+2.1*activity78+.18*treatment78+RNG78.normal(scale=1.7,size=n78); latency78=100+4*activity78+1.2*treatment78+RNG78.normal(scale=8,size=n78)  # 计算并保存当前步骤的中间状态。
sessions78=RNG78.poisson(np.exp(.3+.25*activity78))+1; order_prob78=1/(1+np.exp(-(-1.6+.5*activity78+.06*treatment78))); orders78=RNG78.binomial(sessions78,order_prob78)  # 计算并保存当前步骤的中间状态。
assert all(len(x)==n78 for x in (pre78,revenue78,latency78,sessions78,orders78))  # 用受控断言验证关键不变量。
assert np.corrcoef(pre78,revenue78)[0,1]>.5 and np.all(orders78<=sessions78)  # 用受控断言验证关键不变量。
assert 0<orders78.sum()/sessions78.sum()<1 and np.isfinite(revenue78).all()  # 用受控断言验证关键不变量。

## 4. Difference in means、Welch SE 与置信区间

估计 `mean(T)-mean(C)`，独立组方差为 `s_T²/n_T+s_C²/n_C`。大样本用正态临界值；小样本应使用 t/随机化推断。置信区间表达重复抽样覆盖，不是“真值有 95% 概率在区间”。

同时返回 relative lift，但控制均值接近 0 时应拒绝。

In [ ]:
def diff_means78(values,assignment,alpha=.05):  # 定义本节可复用的核心函数。
    v=np.asarray(values,float); a=np.asarray(assignment)  # 计算并保存当前步骤的中间状态。
    if v.shape!=a.shape or not np.isfinite(v).all() or set(np.unique(a))!={0,1}: raise ValueError("experiment_vector_contract")  # 按当前条件选择后续控制路径。
    c=v[a==0]; t=v[a==1]; effect=float(t.mean()-c.mean()); se=float(math.sqrt(t.var(ddof=1)/len(t)+c.var(ddof=1)/len(c))); z=NORMAL78.inv_cdf(1-alpha/2); return {"control":float(c.mean()),"treatment":float(t.mean()),"effect":effect,"se":se,"ci":(effect-z*se,effect+z*se),"relative":effect/c.mean()}  # 计算并保存当前步骤的中间状态。
raw78=diff_means78(revenue78,treatment78)  # 计算并保存当前步骤的中间状态。
assert raw78["ci"][0]<raw78["effect"]<raw78["ci"][1] and raw78["se"]>0  # 用受控断言验证关键不变量。
assert abs(raw78["effect"]-.18)<.10 and math.isfinite(raw78["relative"])  # 用受控断言验证关键不变量。
assert raw78["control"]>0 and raw78["treatment"]>0  # 用受控断言验证关键不变量。
try: diff_means78([1,2],[0,0]); raise AssertionError("one-arm experiment accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="experiment_vector_contract"  # 捕获预期异常并验证失败分支。

## 5. CUPED：用实验前协变量降低方差

`theta=Cov(Y,X)/Var(X)`，调整 `Y'=Y-theta(X-mean X)`。pre-period 在 treatment 前，不受实验影响；随机化保证两组 pre 均值期望相同。CUPED 不改变 effect 期望，但相关性高时降低 SE。

theta 可在全实验样本估计，因为 treatment 与 pre 独立；更保守可只用历史数据。必须绑定预指标窗口。

In [ ]:
def cuped78(outcome,covariate):  # 定义本节可复用的核心函数。
    y=np.asarray(outcome,float); x=np.asarray(covariate,float)  # 计算并保存当前步骤的中间状态。
    if y.shape!=x.shape or x.var()==0: raise ValueError("cuped_contract")  # 按当前条件选择后续控制路径。
    theta=float(np.cov(y,x,ddof=1)[0,1]/np.var(x,ddof=1)); adjusted=y-theta*(x-x.mean()); return adjusted,theta  # 计算并保存当前步骤的中间状态。
adjusted78,theta78=cuped78(revenue78,pre78); cuped_result78=diff_means78(adjusted78,treatment78)  # 计算并保存当前步骤的中间状态。
assert theta78>0 and cuped_result78["se"]<raw78["se"]*.8  # 用受控断言验证关键不变量。
assert abs(cuped_result78["effect"]-.18)<abs(raw78["effect"]-.18)  # 用受控断言验证关键不变量。
assert abs(cuped_result78["effect"]-raw78["effect"])<.08  # 用受控断言验证关键不变量。
assert math.isclose(adjusted78.mean(),revenue78.mean(),abs_tol=1e-10)  # 用受控断言验证关键不变量。
try: cuped78([1,2],[1,1]); raise AssertionError("constant covariate accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="cuped_contract"  # 捕获预期异常并验证失败分支。

## 6. Ratio metric 的 user cluster delta method

conversion=`Σorders/Σsessions` 不是用户 ratio 的简单平均。对每组估计 ratio，并用 influence `orders-ratio*sessions` 的用户方差除以总 sessions 均值平方，得到 cluster-aware SE；两组再相加。

分母为 0 必须拒绝。heavy users 会主导 ratio，需要预先定义 winsorization/eligibility。

In [ ]:
def ratio_effect78(num,den,assignment):  # 定义本节可复用的核心函数。
    num=np.asarray(num,float); den=np.asarray(den,float); a=np.asarray(assignment); rows=[]  # 计算并保存当前步骤的中间状态。
    for arm in (0,1):  # 遍历输入元素以累积或检查结果。
        n=num[a==arm]; d=den[a==arm]  # 计算并保存当前步骤的中间状态。
        if d.sum()<=0: raise ValueError("ratio_denominator")  # 按当前条件选择后续控制路径。
        ratio=n.sum()/d.sum(); influence=n-ratio*d; var=influence.var(ddof=1)/(len(n)*d.mean()**2); rows.append((float(ratio),float(var)))  # 计算并保存当前步骤的中间状态。
    effect=rows[1][0]-rows[0][0]; se=math.sqrt(rows[0][1]+rows[1][1]); return {"control":rows[0][0],"treatment":rows[1][0],"effect":effect,"se":se,"ci":(effect-1.96*se,effect+1.96*se)}  # 计算并保存当前步骤的中间状态。
ratio78=ratio_effect78(orders78,sessions78,treatment78)  # 计算并保存当前步骤的中间状态。
assert 0<=ratio78["control"]<=1 and 0<=ratio78["treatment"]<=1 and ratio78["se"]>0  # 用受控断言验证关键不变量。
assert ratio78["ci"][0]<ratio78["effect"]<ratio78["ci"][1]  # 用受控断言验证关键不变量。
try: ratio_effect78([0,0],[0,0],[0,1]); raise AssertionError("zero ratio denominator accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="ratio_denominator"  # 捕获预期异常并验证失败分支。

## 7. 多指标、guardrail 与 Holm 校正

同时看很多指标会膨胀假阳性。对主指标、conversion 和 latency 计算双侧 p-value，使用 Holm step-down 控制 family-wise error。guardrail latency 采用非劣界：上置信界若超过允许 +2ms，则不能声明安全。

指标优先级和方向必须实验前注册，不能看到结果后挑赢家。

In [ ]:
def pvalue78(effect,se): return 2*(1-NORMAL78.cdf(abs(effect/se))) if se>0 else 0.  # 定义本节可复用的核心函数。
latency_result78=diff_means78(latency78,treatment78); tests78={"revenue":pvalue78(cuped_result78["effect"],cuped_result78["se"]),"conversion":pvalue78(ratio78["effect"],ratio78["se"]),"latency":pvalue78(latency_result78["effect"],latency_result78["se"])}  # 计算并保存当前步骤的中间状态。
def holm78(pvalues,alpha=.05):  # 定义本节可复用的核心函数。
    ordered=sorted(pvalues.items(),key=lambda z:(z[1],z[0])); rejected={k:False for k in pvalues}; active=True  # 计算并保存当前步骤的中间状态。
    for i,(name,p) in enumerate(ordered):  # 遍历输入元素以累积或检查结果。
        if active and p<=alpha/(len(ordered)-i): rejected[name]=True  # 按当前条件选择后续控制路径。
        else: active=False  # 计算并保存当前步骤的中间状态。
    return rejected  # 返回当前分支计算出的结果。
rejected78=holm78(tests78); latency_upper78=latency_result78["ci"][1]; guardrail_safe78=latency_upper78<2.  # 计算并保存当前步骤的中间状态。
assert set(rejected78)==set(tests78) and all(0<=p<=1 for p in tests78.values())  # 用受控断言验证关键不变量。
assert latency_result78["effect"]>0 and guardrail_safe78==bool(latency_upper78<2.)  # 用受控断言验证关键不变量。
assert holm78({"a":.001,"b":.9})=={"a":True,"b":False}  # 用受控断言验证关键不变量。

## 8. 提前偷看与 alpha spending

每天反复用 0.05 阈值检验会累积假阳性。若必须 sequential monitoring，要预先定义 look 次数和 alpha spending。这里用 Bonferroni `alpha/looks` 的保守边界，对 5 个累计 look 计算 z；决策只在跨越预注册边界时触发。

更高效可用 group sequential/O'Brien-Fleming、always-valid p/e-value，但不能事后补规则。

In [ ]:
order78=np.argsort([bucket78(u,"analysis-order") for u in users78]); looks78=5; look_rows78=[]; boundary78=NORMAL78.inv_cdf(1-.05/(2*looks78))  # 计算并保存当前步骤的中间状态。
for frac in np.linspace(.2,1.,looks78):  # 遍历输入元素以累积或检查结果。
    idx=order78[:int(n78*frac)]; result=diff_means78(adjusted78[idx],treatment78[idx]); z=result["effect"]/result["se"]; look_rows78.append((len(idx),z,abs(z)>=boundary78))  # 计算并保存当前步骤的中间状态。
assert len(look_rows78)==looks78 and boundary78>1.96  # 用受控断言验证关键不变量。
assert all(look_rows78[i][0]<look_rows78[i+1][0] for i in range(looks78-1))  # 用受控断言验证关键不变量。
assert all(math.isfinite(z) for _,z,_ in look_rows78)  # 用受控断言验证关键不变量。

## 9. 实验注册、审计与来源

manifest 绑定实验 ID/salt、eligibility、ratio、随机化/分析单位、指标公式、CUPED 窗口、样本量、multiple-testing、sequential rule、guardrail 和数据摘要。分析从实际用户级数组重算 digest，防止换样本后沿用结论。

面试回答顺序：假设/单位 → 分桶/SRM → 指标与样本量 → estimator/CUPED → guardrail/多重 → sequential → rollout。任何 SRM 或日志缺失先阻塞效果解释。

In [ ]:
data_digest78=sha78(np.ascontiguousarray(np.column_stack([treatment78,pre78,revenue78,latency78,sessions78,orders78])).tobytes())  # 计算并保存当前步骤的中间状态。
manifest78={"artifact_id":"search-ui-v1","salt":"s1","allocation":.5,"unit":"user","eligibility":"controlled-all-users","primary":"mean_revenue","cuped":{"covariate":"pre_revenue","theta":theta78},"ratio":"sum_orders/sum_sessions user-cluster-delta","multiple":"Holm FWER .05","sequential":{"looks":looks78,"boundary":boundary78,"method":"Bonferroni spending"},"guardrail":{"latency_noninferiority_ms":2.},"data_digest":data_digest78,"n":n78}  # 计算并保存当前步骤的中间状态。
TRUST78=MappingProxyType({manifest78["artifact_id"]:sha78(canonical78(manifest78).encode())})  # 计算并保存当前步骤的中间状态。
def load_experiment78(m,arrays):  # 定义本节可复用的核心函数。
    actual=copy.deepcopy(m); actual["data_digest"]=sha78(np.ascontiguousarray(np.column_stack(arrays)).tobytes()); actual["n"]=len(arrays[0])  # 计算并保存当前步骤的中间状态。
    if TRUST78.get(actual.get("artifact_id"))!=sha78(canonical78(actual).encode()): raise RuntimeError("untrusted_experiment_snapshot")  # 按当前条件选择后续控制路径。
    return MappingProxyType(actual)  # 返回当前分支计算出的结果。
published78=load_experiment78(manifest78,[treatment78,pre78,revenue78,latency78,sessions78,orders78])  # 计算并保存当前步骤的中间状态。
assert published78["n"]==12000 and isinstance(TRUST78,MappingProxyType)  # 用受控断言验证关键不变量。
forged_revenue78=revenue78.copy(); forged_revenue78[0]+=100  # 计算并保存当前步骤的中间状态。
try: load_experiment78(manifest78,[treatment78,pre78,forged_revenue78,latency78,sessions78,orders78]); raise AssertionError("forged experiment accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="untrusted_experiment_snapshot"  # 捕获预期异常并验证失败分支。
print({"srm_p":round(srm_ok78["p"],3),"raw_effect":round(raw78["effect"],3),"cuped_effect":round(cuped_result78["effect"],3),"se_reduction":round(1-cuped_result78["se"]/raw78["se"],3),"latency_upper":round(latency_upper78,3)})  # 执行当前语句以推进本节示例。

## 10. 失败模式与进一步追问

常见错误：请求级随机、SRM 后继续读效果、事件行当独立样本、ratio 平均用户比率、CUPED 用实验后变量、多指标挑显著、每天偷看 0.05、guardrail 只看点估计、分桶 salt 变更和实验数据快照不可复现。还应讨论异质性、网络效应、长期效应与 novelty。

- Deng et al., [Improving the Sensitivity of Online Controlled Experiments by Utilizing Pre-Experiment Data](https://www.microsoft.com/en-us/research/wp-content/uploads/2013/02/KDD2013-ExP.pdf)，CUPED。
- Kohavi et al., [Trustworthy Online Controlled Experiments](https://experimentguide.com/)，工程方法背景。
- Johari et al., [Always Valid Inference](https://arxiv.org/abs/1512.04922)，sequential inference。